# Asian Options & Control Variates

Pricing an arithmetic-average Asian call by Monte Carlo, using the geometric-average
Asian as a control variate for variance reduction.

## Asian options

Payoff depends on the *average* price over monitoring dates $t_i = iT/n$, not the
terminal price $S_T$ - so the payoff is **path-dependent** and requires step-by-step
path simulation (unlike a European, priced from $S_T$ alone).

$$Y_{\text{arith}} = \left(\bar S_{\text{arith}} - K\right)^+, \quad \bar S_{\text{arith}} = \frac1n\sum_{i=1}^n S_{t_i}$$

$$Y_{\text{geo}} = \left(\bar S_{\text{geo}} - K\right)^+, \quad \bar S_{\text{geo}} = \left(\prod_{i=1}^n S_{t_i}\right)^{1/n}$$

The arithmetic average has **no closed form** (sum of lognormals is not lognormal),
forcing MC. The geometric average *is* lognormal (sum of normals in log-space),
giving a Black-Scholes-style closed form - this is what makes it a usable control.

## Geometric-Asian closed form

$\log\bar S_{\text{geo}} = \frac1n\sum_i \log S_{t_i}$ is normal. Its variance needs
the shared-path covariance $\text{Cov}(\log S_{t_i}, \log S_{t_j}) = \sigma^2\min(t_i,t_j)$
(the log-prices are NOT independent - they share Brownian history). Working the double
sum with $\sum_i\sum_j\min(i,j) = \frac{n(n+1)(2n+1)}{6}$ gives effective parameters:

$$\hat\sigma = \sigma\sqrt{\frac{(n+1)(2n+1)}{6n^2}} \;\xrightarrow{n\to\infty}\; \frac{\sigma}{\sqrt3}, \qquad \bar t = \frac{T(n+1)}{2n}$$

$$\hat b = \frac{(r - \tfrac12\sigma^2)\bar t + \tfrac12\hat\sigma^2 T}{T}$$

Price as a BS call with vol $\hat\sigma$, forward grown at $\hat b$, discounted at $r$:

$$\mu_X = e^{-rT}\big[S_0 e^{\hat b T}\Phi(\hat d_1) - K\Phi(\hat d_2)\big], \quad \hat d_1 = \frac{\log(S_0/K) + (\hat b + \tfrac12\hat\sigma^2)T}{\hat\sigma\sqrt T}, \quad \hat d_2 = \hat d_1 - \hat\sigma\sqrt T$$

Geometric avg is *less volatile* than the underlying ($\hat\sigma < \sigma$), so the
geometric Asian is cheaper than the vanilla call - the cheap directional sanity check.

## Control variates

For target $Y$ (unknown mean) and control $X$ (known mean $\mu_X$), correlated:

$$Y_{\text{cv}} = Y - c\,(X - \mu_X)$$

Unbiased for any $c$ (the correction has mean zero). Optimal $c$ minimises the variance:

$$c^* = \frac{\text{Cov}(X,Y)}{\text{Var}(X)} \quad\text{(the OLS slope of $Y$ on $X$)}, \qquad \text{Var}(Y_{\text{cv}}^*) = \text{Var}(Y)\,(1-\rho^2)$$

Reduction depends on $\rho^2$ only (sign of $\rho$ irrelevant). Strongly nonlinear:
$\rho=0.9 \to 5\times$, $\rho=0.99 \to 50\times$. The skill is *choosing* a high-$\rho$ control. A control variate is a known-answer rehearsal of the same noise, the better it rehearses (higher $\rho$), the more of your unknown's uncertainty it cancels.

In [1]:
import sys
from scipy.stats import norm
import numpy as np
sys.path.append('..')  # so we can import from models/
from models.exotics import geometric_asian_price, arithmetic_asian_price_cv
from models.montecarlo import simulate_gbm_paths
from models.bsm import bsm_price

In [2]:
print("Vanilla price: ", bsm_price(100,100,1,0.05,0.2,'call'))
print("Asian price: ", geometric_asian_price(100,100,1,0.05,0.2, 12))

Vanilla price:  10.450583572185565
Asian price:  5.9402002216335


In [3]:
S, K, T, r, sigma, n_steps = 100.0, 100.0, 1.0, 0.05, 0.20, 12
paths = simulate_gbm_paths(S, T, r, sigma, n_steps, n_paths=1_000_000, seed=42)

# geometric average per path: exp(mean of log along TIME axis)
geo_mean = np.exp(np.mean(np.log(paths), axis=1))     # -> (n_paths,)
payoff   = np.maximum(geo_mean - K, 0.0)
disc_payoff = np.exp(-r * T) * payoff

mc_price = disc_payoff.mean()
mc_se    = disc_payoff.std(ddof=1) / np.sqrt(len(disc_payoff))

closed = geometric_asian_price(S, K, T, r, sigma, n_steps)
z = abs(mc_price - closed) / mc_se
print(f"MC = {mc_price:.5f} ± {mc_se:.5f}   closed = {closed:.5f}   z = {z:.2f}")

MC = 5.94384 ± 0.00825   closed = 5.94020   z = 0.44


In [4]:
# Control Variate test
print(arithmetic_asian_price_cv(S, K, T, r, sigma, n_steps, n_paths=100_000, seed=None))

{'plain': (np.float64(6.144365149096251), np.float64(0.026948508708788833)), 'cv': (np.float64(6.1563710009961845), np.float64(0.0007583341198354026)), 'c_star': np.float64(1.0316333479616515), 'corr': np.float64(0.9996039885663092)}


# Lessons

## The reduction formula holds exactly

Measured variance ratio $(\text{cv\_se}/\text{plain\_se})^2 \approx 7.8\times10^{-4}$
matched the predicted $1 - \rho^2 = 1 - 0.99961^2 \approx 7.8\times10^{-4}$ to the
digit. The theory is not approximate - the OLS-slope $c^*$ delivers exactly
$\text{Var}(Y)(1-\rho^2)$.

## Why the geometric control is so good

$\rho \approx 0.9996$ because the arithmetic and geometric averages of the *same path*
differ only by the (tiny, near-constant) AM-GM gap of tightly-clustered prices. The two
payoffs move in near-lockstep. This is *the* textbook control variate precisely because
$\rho$ is pathologically close to 1 - most real controls are far weaker.

Supporting checks: $c^* \approx 1.03$ (≈1, since $X,Y$ near-identical magnitude and
near-perfectly correlated); plain and CV point estimates agree within the error bar
(CV is unbiased, only shrinks noise).

## Variance reduction buys compute, not rate

CV gave ~36x SE reduction but did NOT change the $O(N^{-1/2})$ rate - same lesson as
the Greeks notebook. To match that 36x by brute force needs $36^2 \approx 1300\times$
the paths. So one closed-form formula + a covariance estimate bought a ~1300x compute
speedup at fixed accuracy. That is the entire economic case for variance reduction:
cheap analytic insight buying compute you'd otherwise pay for at the punishing
$N^{-1/2}$ exchange rate.

## Method note: estimating $c^*$ from the sample

$c^*$ is estimated from the same paths it is applied to (uses the data twice),
introducing a small bias - negligible at $10^5$+ paths. The clean $(1-\rho^2)$ is the
best case; pilot-run or regression estimates of $c^*$ remove the reuse bias if needed.

## Code structure

- `simulate_gbm_paths` (montecarlo.py): general path primitive, exact log-space
  stepping, `(n_paths, n_steps)`, validated per-timestep. Reused for all path-dependent
  payoffs (barriers, stochastic vol W4-9).
- `geometric_asian_price`, `arithmetic_asian_cv` (exotics.py): the instrument and its
  CV estimator. Axis discipline: noise accumulates along time (`axis=1`), statistics
  aggregate over paths (`axis=0`); geometric = `exp(mean(log))`, arithmetic = `mean`.